In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# 1. 指定Mac系统自带的Arial Unicode MS字体路径（固定路径，无需修改）
font_path = '/Library/Fonts/Arial Unicode.ttf'

# 2. 创建字体对象（size可根据需求调整）
mac_font = fm.FontProperties(fname=font_path, size=11)

# 读取数据
file_path = '/Users/inaya/Desktop/HierSales/金佰利数据处理结果_20260304_130926_透视表_转置结果.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 数据基本信息探索
print("=" * 60)
print("数据集基本信息")
print("=" * 60)
print(f"数据形状: {df.shape} (行数, 列数)")
print(f"\n列名列表:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\n数据类型:")
print(df.dtypes)

print(f"\n前5行数据预览:")
print(df.head())

print(f"\n缺失值统计:")
missing_stats = df.isnull().sum()
missing_stats = missing_stats[missing_stats > 0]
if len(missing_stats) > 0:
    print(missing_stats)
else:
    print("无缺失值")

print(f"\n数值型数据统计描述:")
print(df.describe())

数据集基本信息
数据形状: (2493, 18) (行数, 列数)

列名列表:
 1. 月份
 2. Y22-01
 3. Y22-02
 4. Y22-03
 5. Y22-04
 6. Y22-05
 7. Y22-06
 8. Y22-07
 9. Y22-08
10. Y22-09
11. Y22-10
12. Y22-11
13. Y22-12
14. 地城市
15. 大区
16. 小区
17. 门店编码
18. 渠道

数据类型:
月份            str
Y22-01    float64
Y22-02    float64
Y22-03    float64
Y22-04    float64
Y22-05    float64
Y22-06    float64
Y22-07    float64
Y22-08    float64
Y22-09    float64
Y22-10    float64
Y22-11    float64
Y22-12    float64
地城市           str
大区            str
小区            str
门店编码          str
渠道            str
dtype: object

前5行数据预览:
                                   月份      Y22-01      Y22-02      Y22-03  \
0  (3046)灌云人民路华润苏果购物广场(807001)???购物广场  32227.4524  17094.3306  30028.2642   
1                     (中山店)昆山润华商业有限公司  37963.2894  22404.0926  60672.7204   
2                     (北海店)昆山润华商业有限公司  66162.2928  29020.2091  81687.8872   
3                     (厚街店)昆山润华商业有限公司  58619.9745  34768.5532  81703.1778   
4                     (厦门店)昆山润华商业有限公司  498

In [12]:
# 先分析渠道和地城市的分布情况
print("=" * 60)
print("渠道分布统计")
print("=" * 60)
channel_counts = df['渠道'].value_counts()
channel_percent = df['渠道'].value_counts(normalize=True) * 100
channel_stats = pd.DataFrame({
    '门店数量': channel_counts,
    '占比(%)': channel_percent.round(2)
})
print(channel_stats)

print("\n" + "=" * 60)
print("地城市分布统计 (前20个城市)")
print("=" * 60)
city_counts = df['地城市'].value_counts()
city_percent = df['地城市'].value_counts(normalize=True) * 100
city_stats = pd.DataFrame({
    '门店数量': city_counts,
    '占比(%)': city_percent.round(2)
})
print(city_stats.head(20))

print(f"\n地城市总数: {len(city_counts)} 个")
print(f"渠道总数: {len(channel_counts)} 个")

# 计算各渠道的月度销售总和，用于后续可视化
month_columns = [col for col in df.columns if col.startswith('Y22-')]
df['年度销售总和'] = df[month_columns].sum(axis=1)

# 按渠道分组计算销售统计
channel_sales = df.groupby('渠道')['年度销售总和'].agg(['sum', 'mean', 'count']).round(2)
channel_sales.columns = ['年度销售总额', '单店平均销售额', '门店数量']
channel_sales['年度销售总额(万元)'] = (channel_sales['年度销售总额'] / 10000).round(2)
channel_sales['单店平均销售额(万元)'] = (channel_sales['单店平均销售额'] / 10000).round(2)
channel_sales = channel_sales.sort_values('年度销售总额', ascending=False)

print("\n" + "=" * 60)
print("各渠道销售业绩统计")
print("=" * 60)
print(channel_sales[['门店数量', '年度销售总额(万元)', '单店平均销售额(万元)']])

渠道分布统计
                  门店数量  占比(%)
渠道                           
永辉                 476  19.09
大润发                414  16.61
沃尔玛                285  11.43
华润                 173   6.94
GT-MT              150   6.02
...                ...    ...
玉溪大尔多                1   0.04
GT-Others, 经销商       1   0.04
太原朝批商超美特好, GT-MT     1   0.04
香百, GT-MT            1   0.04
信誉楼, 沧州劲草            1   0.04

[83 rows x 2 columns]

地城市分布统计 (前20个城市)
     门店数量  占比(%)
地城市             
北京    172   6.90
上海    111   4.45
深圳     99   3.97
杭州     73   2.93
广州     66   2.65
南京     62   2.49
重庆     61   2.45
武汉     54   2.17
成都     47   1.89
佛山     45   1.81
中山     44   1.76
天津     44   1.76
东莞     42   1.68
青岛     42   1.68
苏州     40   1.60
合肥     37   1.48
常州     34   1.36
无锡     32   1.28
洛阳     28   1.12
济南     28   1.12

地城市总数: 392 个
渠道总数: 83 个

各渠道销售业绩统计
          门店数量  年度销售总额(万元)  单店平均销售额(万元)
渠道                                     
大润发        414    18808.65        45.43
沃尔玛        285    13126.81      

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# --------------------------
# Mac系统专属配置
# --------------------------
font_path = '/Library/Fonts/Arial Unicode.ttf'
mac_font = fm.FontProperties(fname=font_path, size=11)
plt.rcParams['axes.unicode_minus'] = False
plt.switch_backend('TkAgg')

# --------------------------
# 数据预处理
# --------------------------
file_path = '金佰利数据处理结果_20260304_130926_透视表_转置结果.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

channel_counts = df['渠道'].value_counts()
city_counts = df['地城市'].value_counts()

month_columns = [col for col in df.columns if col.startswith('Y22-')]
df['年度销售总和'] = df[month_columns].sum(axis=1)

channel_sales = df.groupby('渠道')['年度销售总和'].agg(['sum', 'mean', 'count']).round(2)
channel_sales.columns = ['年度销售总额', '单店平均销售额', '门店数量']
channel_sales['年度销售总额(万元)'] = (channel_sales['年度销售总额'] / 10000).round(2)
channel_sales['单店平均销售额(万元)'] = (channel_sales['单店平均销售额'] / 10000).round(2)
channel_sales = channel_sales.sort_values('年度销售总额', ascending=False)

# --------------------------
# 可视化配置（舒展版布局）
# --------------------------
plt.style.use('default')
colors_channel = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', 
                  '#577590', '#F8961E', '#90A959', '#F9C74F', '#90BE6D']

# 扩大画布尺寸
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 14))

fig.suptitle('金佰利渠道与城市分布统计分析', fontproperties=mac_font, 
             fontsize=22, fontweight='bold', y=0.98)

# --------------------------
# 子图1：渠道门店数量分布（前10）
# --------------------------
top10_channels = channel_counts.head(10)
bars1 = ax1.bar(range(len(top10_channels)), top10_channels.values, 
                color=colors_channel[:len(top10_channels)], alpha=0.8, 
                edgecolor='white', linewidth=1.2)

ax1.set_title('各渠道门店数量分布（前10）', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('渠道名称', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax1.set_ylabel('门店数量', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax1.set_xticks(range(len(top10_channels)))
ax1.set_xticklabels(top10_channels.index, fontproperties=mac_font, 
                    rotation=45, ha='right', fontsize=11)
for label in ax1.get_yticklabels():
    label.set_fontproperties(mac_font)

for i, bar in enumerate(bars1):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 5,
             f'{int(height)}', ha='center', va='bottom', 
             fontproperties=mac_font, fontweight='bold', fontsize=10)

ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_ylim(0, max(top10_channels.values) * 1.1)

# --------------------------
# 子图2：渠道销售总额分布（前10）
# --------------------------
top10_sales_channels = channel_sales.head(10)
bars2 = ax2.bar(range(len(top10_sales_channels)), top10_sales_channels['年度销售总额(万元)'],
                color=colors_channel[:len(top10_sales_channels)], alpha=0.8, 
                edgecolor='white', linewidth=1.2)

ax2.set_title('各渠道年度销售总额分布（前10）', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('渠道名称', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax2.set_ylabel('年度销售总额（万元）', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax2.set_xticks(range(len(top10_sales_channels)))
ax2.set_xticklabels(top10_sales_channels.index, fontproperties=mac_font, 
                    rotation=45, ha='right', fontsize=11)
for label in ax2.get_yticklabels():
    label.set_fontproperties(mac_font)

for i, bar in enumerate(bars2):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 200,
             f'{int(height)}', ha='center', va='bottom', 
             fontproperties=mac_font, fontweight='bold', fontsize=10)

ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_ylim(0, max(top10_sales_channels['年度销售总额(万元)']) * 1.1)

# --------------------------
# 子图3：渠道单店平均销售额（前10）
# --------------------------
top10_avg_sales = channel_sales.sort_values('单店平均销售额(万元)', ascending=False).head(10)
bars3 = ax3.bar(range(len(top10_avg_sales)), top10_avg_sales['单店平均销售额(万元)'],
                color=colors_channel[:len(top10_avg_sales)], alpha=0.8, 
                edgecolor='white', linewidth=1.2)

ax3.set_title('各渠道单店平均销售额分布（前10）', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax3.set_xlabel('渠道名称', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax3.set_ylabel('单店平均销售额（万元）', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax3.set_xticks(range(len(top10_avg_sales)))
ax3.set_xticklabels(top10_avg_sales.index, fontproperties=mac_font, 
                    rotation=45, ha='right', fontsize=11)
for label in ax3.get_yticklabels():
    label.set_fontproperties(mac_font)

for i, bar in enumerate(bars3):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.1f}', ha='center', va='bottom', 
             fontproperties=mac_font, fontweight='bold', fontsize=10)

ax3.grid(axis='y', alpha=0.3, linestyle='--')
ax3.set_ylim(0, max(top10_avg_sales['单店平均销售额(万元)']) * 1.1)

# --------------------------
# 子图4：门店数量城市分布（前15）
# --------------------------
top15_cities = city_counts.head(15)
bars4 = ax4.barh(range(len(top15_cities)), top15_cities.values,
                 color=colors_channel[:len(top15_cities)], alpha=0.8, 
                 edgecolor='white', linewidth=1.2)

ax4.set_title('各城市门店数量分布（前15）', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax4.set_xlabel('门店数量', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax4.set_ylabel('城市名称', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax4.set_yticks(range(len(top15_cities)))
ax4.set_yticklabels(top15_cities.index, fontproperties=mac_font, fontsize=11)
for label in ax4.get_xticklabels():
    label.set_fontproperties(mac_font)

for i, bar in enumerate(bars4):
    width = bar.get_width()
    ax4.text(width + 2, bar.get_y() + bar.get_height()/2.,
             f'{int(width)}', ha='left', va='center', 
             fontproperties=mac_font, fontweight='bold', fontsize=10)

ax4.grid(axis='x', alpha=0.3, linestyle='--')
ax4.set_xlim(0, max(top15_cities.values) * 1.1)

# --------------------------
# 舒展化布局调整
# --------------------------
plt.tight_layout()
plt.subplots_adjust(
    top=0.94,
    bottom=0.12,
    left=0.08,
    right=0.92,
    hspace=0.35,  # 上下子图间距加大
    wspace=0.25   # 左右子图间距加大
)

# 保存高清图片
save_path = '金佰利渠道城市分布分析_舒展版.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.close()

print("✅ 舒展版图表已生成完成！")
print(f"📁 保存路径: {save_path}")

✅ 舒展版图表已生成完成！
📁 保存路径: 金佰利渠道城市分布分析_舒展版.png


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# --------------------------
# Mac系统专属配置（核心：中文显示+后端兼容）
# --------------------------
# 1. 指定Mac自带中文字体路径
font_path = '/Library/Fonts/Arial Unicode.ttf'
# 2. 创建字体对象（size=11适配图表，可微调）
mac_font = fm.FontProperties(fname=font_path, size=11)
# 3. 解决负号显示异常
plt.rcParams['axes.unicode_minus'] = False
# 4. 避免tostring_rgb错误，指定Mac兼容后端
plt.switch_backend('TkAgg')

# --------------------------
# 数据预处理（保持原分析逻辑）
# --------------------------
# 读取数据（根据实际路径调整）
file_path = '金佰利数据处理结果_20260304_130926_透视表_转置结果.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')

# 基础统计量计算
channel_counts = df['渠道'].value_counts()
city_counts = df['地城市'].value_counts()
channel_percent = df['渠道'].value_counts(normalize=True) * 100

# 计算年度销售总和及渠道销售统计
month_columns = [col for col in df.columns if col.startswith('Y22-')]
df['年度销售总和'] = df[month_columns].sum(axis=1)

channel_sales = df.groupby('渠道')['年度销售总和'].agg(['sum', 'mean', 'count']).round(2)
channel_sales.columns = ['年度销售总额', '单店平均销售额', '门店数量']
channel_sales['年度销售总额(万元)'] = (channel_sales['年度销售总额'] / 10000).round(2)
channel_sales['单店平均销售额(万元)'] = (channel_sales['单店平均销售额'] / 10000).round(2)
channel_sales = channel_sales.sort_values('年度销售总额', ascending=False)

# 1. 按大区统计门店分布和销售情况
region_stats = df.groupby('大区').agg({
    '门店编码': 'count',
    '年度销售总和': ['sum', 'mean']
}).round(2)

region_stats.columns = ['门店数量', '年度销售总额', '单店平均销售额']
region_stats['年度销售总额(万元)'] = (region_stats['年度销售总额'] / 10000).round(2)
region_stats['单店平均销售额(万元)'] = (region_stats['单店平均销售额'] / 10000).round(2)
region_stats = region_stats.sort_values('门店数量', ascending=False)

print("=" * 60)
print("各大区门店与销售统计")
print("=" * 60)
print(region_stats[['门店数量', '年度销售总额(万元)', '单店平均销售额(万元)']])

# 2. 创建城市-渠道交叉分析（选择主要城市和渠道）
major_cities = city_counts.head(10).index.tolist()
major_channels = channel_counts.head(5).index.tolist()

# 筛选主要城市和渠道的数据
city_channel_data = df[df['地城市'].isin(major_cities) & df['渠道'].isin(major_channels)]

# 创建交叉表
city_channel_pivot = pd.pivot_table(
    city_channel_data,
    values='年度销售总和',
    index='地城市',
    columns='渠道',
    aggfunc='sum',
    fill_value=0
)

# 转换为万元
city_channel_pivot = (city_channel_pivot / 10000).round(2)

print("\n" + "=" * 60)
print("主要城市-渠道销售交叉分析（单位：万元）")
print("=" * 60)
print(city_channel_pivot)

# --------------------------
# 可视化配置（所有中文文本指定mac_font，优化舒展布局）
# --------------------------
# 扩大画布尺寸，避免紧凑
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16))
# 全局标题（中文+加粗+适配字体）
fig.suptitle('金佰利大区分布与城市渠道交叉分析', fontproperties=mac_font, 
             fontsize=22, fontweight='bold', y=0.98)

# 定义颜色映射
cmap1 = mcolors.LinearSegmentedColormap.from_list("custom_blue", ["#E6F3FF", "#1E88E5", "#0D47A1"])
cmap2 = mcolors.LinearSegmentedColormap.from_list("custom_green", ["#E8F5E9", "#4CAF50", "#1B5E20"])

# --------------------------
# 子图1：各大区门店数量分布
# --------------------------
bars1 = ax1.bar(region_stats.index, region_stats['门店数量'], 
                color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'],
                alpha=0.8, edgecolor='white', linewidth=1.5)

# 标题（中文+字体指定）
ax1.set_title('各大区门店数量分布', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
# 轴标签（增加labelpad避免重叠）
ax1.set_xlabel('大区', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax1.set_ylabel('门店数量', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
# X轴刻度（旋转45度+中文适配）
ax1.tick_params(axis='x', rotation=45)
ax1.set_xticklabels(region_stats.index, fontproperties=mac_font, fontsize=11)
# Y轴刻度（中文适配）
for label in ax1.get_yticklabels():
    label.set_fontproperties(mac_font)

# 数值标签（中文字体+清晰显示）
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 10,
             f'{int(height)}', ha='center', va='bottom', 
             fontproperties=mac_font, fontweight='bold', fontsize=10)

ax1.grid(axis='y', alpha=0.3, linestyle='--')
# 增加Y轴缓冲，避免数值标签被截断
ax1.set_ylim(0, max(region_stats['门店数量']) * 1.15)

# --------------------------
# 子图2：各大区销售总额分布
# --------------------------
bars2 = ax2.bar(region_stats.index, region_stats['年度销售总额(万元)'],
                color=['#FF8A80', '#80CBC4', '#80D8F0', '#C8E6C9'],
                alpha=0.8, edgecolor='white', linewidth=1.5)

ax2.set_title('各大区年度销售总额分布', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('大区', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax2.set_ylabel('年度销售总额（万元）', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax2.tick_params(axis='x', rotation=45)
ax2.set_xticklabels(region_stats.index, fontproperties=mac_font, fontsize=11)
for label in ax2.get_yticklabels():
    label.set_fontproperties(mac_font)

# 数值标签
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 500,
             f'{int(height)}', ha='center', va='bottom', 
             fontproperties=mac_font, fontweight='bold', fontsize=10)

ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_ylim(0, max(region_stats['年度销售总额(万元)']) * 1.15)

# --------------------------
# 子图3：主要城市-渠道销售热力图（重点中文适配）
# --------------------------
im = ax3.imshow(city_channel_pivot.values, cmap=cmap1, aspect='auto')

# 标题+轴标签（中文适配）
ax3.set_title('主要城市-渠道年度销售热力图（万元）', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax3.set_xlabel('渠道', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax3.set_ylabel('城市', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)

# 刻度标签（渠道+城市中文适配）
ax3.set_xticks(range(len(city_channel_pivot.columns)))
ax3.set_xticklabels(city_channel_pivot.columns, fontproperties=mac_font, 
                    rotation=45, ha='right', fontsize=10)
ax3.set_yticks(range(len(city_channel_pivot.index)))
ax3.set_yticklabels(city_channel_pivot.index, fontproperties=mac_font, fontsize=10)

# 热力图数值标注（中文字体+清晰显示）
for i in range(len(city_channel_pivot.index)):
    for j in range(len(city_channel_pivot.columns)):
        value = city_channel_pivot.iloc[i, j]
        # 根据数值调整文字颜色，确保在深浅背景上都清晰
        text_color = "white" if value > city_channel_pivot.values.max() * 0.6 else "black"
        text = ax3.text(j, i, f'{value:.0f}',
                       ha="center", va="center", color=text_color,
                       fontproperties=mac_font, fontweight='bold', fontsize=9)

# 颜色条标签（中文适配）
cbar1 = plt.colorbar(im, ax=ax3, shrink=0.8)
cbar1.set_label('年度销售总额（万元）', fontproperties=mac_font, 
                rotation=270, labelpad=25, fontweight='bold', fontsize=10)
# 颜色条刻度中文适配
for label in cbar1.ax.get_yticklabels():
    label.set_fontproperties(mac_font)

# --------------------------
# 子图4：主要渠道月度销售趋势（中文适配）
# --------------------------
major_channel_data = df[df['渠道'].isin(major_channels)]
monthly_trend = major_channel_data.groupby('渠道')[month_columns].mean()
# 转换为万元
monthly_trend = (monthly_trend / 10000).round(2)

# 绘制趋势线
colors_trend = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for i, (channel, color) in enumerate(zip(monthly_trend.index, colors_trend)):
    ax4.plot(range(1, 13), monthly_trend.loc[channel], 
             marker='o', linewidth=2.5, markersize=6, 
             label=channel, color=color, alpha=0.8)

ax4.set_title('主要渠道月度平均销售额趋势', fontproperties=mac_font, 
              fontsize=14, fontweight='bold', pad=20)
ax4.set_xlabel('月份', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
ax4.set_ylabel('月均销售额（万元）', fontproperties=mac_font, fontsize=12, fontweight='bold', labelpad=12)
# 月份刻度（01-12格式+中文适配）
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels([f'{i:02d}' for i in range(1, 13)], fontproperties=mac_font, fontsize=10)
for label in ax4.get_yticklabels():
    label.set_fontproperties(mac_font)

# 图例（中文适配+清晰布局）
ax4.legend(loc='upper right', frameon=True, fancybox=True, shadow=True,
           prop=mac_font, fontsize=10, bbox_to_anchor=(1, 1))

ax4.grid(True, alpha=0.3, linestyle='--')
# 增加Y轴缓冲，避免趋势线顶部超出
ax4.set_ylim(0, monthly_trend.values.max() * 1.2)

# --------------------------
# 舒展化布局调整（核心解决紧凑问题）
# --------------------------
plt.tight_layout()
plt.subplots_adjust(
    top=0.94,    # 全局标题与子图顶部距离
    bottom=0.15, # 底部边距（防止X轴标签被截断）
    left=0.10,   # 左侧边距
    right=0.90,  # 右侧边距
    hspace=0.40, # 上下子图垂直间距（加大至0.4）
    wspace=0.30  # 左右子图水平间距（加大至0.3）
)

# 保存高分辨率图片（Mac路径适配）
save_path = '金佰利大区城市渠道深度分析_中文舒展版.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.close()

print("\n✅ 深度分析图表（中文舒展版）已生成完成！")
print(f"📁 深度分析图表保存路径: {save_path}")

# --------------------------
# 生成统计报告（保持原逻辑）
# --------------------------
report_content = f"""# 金佰利数据统计分析报告

## 1. 数据概览
- **数据规模**: 共{df.shape[0]:,}条门店记录，{df.shape[1]}个字段
- **时间范围**: 2022年1-12月
- **地理覆盖**: {len(city_counts)}个城市，{len(df['大区'].unique())}个大区
- **渠道数量**: {len(channel_counts)}个不同渠道

## 2. 渠道分析总结

### 2.1 门店数量分布（前5名）
"""

# 添加渠道排名
for i, (channel, count) in enumerate(channel_counts.head(5).items(), 1):
    percent = channel_percent[channel]
    report_content += f"{i}. **{channel}**: {count}家门店 ({percent:.1f}%)\n"

report_content += f"""
### 2.2 销售业绩排名（前5名）
"""

# 添加销售排名
for i, (channel, data) in enumerate(channel_sales.head(5).iterrows(), 1):
    sales = data['年度销售总额(万元)']
    avg_sales = data['单店平均销售额(万元)']
    report_content += f"{i}. **{channel}**: 总销售额{sales:.0f}万元，单店平均{avg_sales:.1f}万元\n"

report_content += f"""
### 2.3 单店效率排名（前5名）
"""

# 添加单店效率排名
top5_efficiency = channel_sales.sort_values('单店平均销售额(万元)', ascending=False).head(5)
for i, (channel, data) in enumerate(top5_efficiency.iterrows(), 1):
    avg_sales = data['单店平均销售额(万元)']
    total_sales = data['年度销售总额(万元)']
    report_content += f"{i}. **{channel}**: 单店平均{avg_sales:.1f}万元，总销售额{total_sales:.0f}万元\n"

report_content += f"""
## 3. 地理分布分析

### 3.1 城市门店分布（前10名）
"""

# 添加城市排名
for i, (city, count) in enumerate(city_counts.head(10).items(), 1):
    percent = city_percent[city]
    report_content += f"{i}. **{city}**: {count}家门店 ({percent:.1f}%)\n"

report_content += f"""
### 3.2 大区分布统计
"""

# 添加大区统计
for region, data in region_stats.iterrows():
    report_content += f"- **{region}**: {data['门店数量']}家门店，总销售额{data['年度销售总额(万元)']:.0f}万元，单店平均{data['单店平均销售额(万元)']:.1f}万元\n"

report_content += f"""
## 4. 关键发现与洞察

1. **渠道集中度高**: 前3大渠道（永辉、大润发、沃尔玛）占据{channel_counts.head(3).sum()/channel_counts.sum()*100:.1f}%的门店资源
2. **销售效率差异大**: 最高单店平均销售额（{top5_efficiency.iloc[0].name}）是最低的{top5_efficiency.iloc[0]['单店平均销售额(万元)']/channel_sales['单店平均销售额(万元)'].min():.0f}倍
3. **城市布局集中**: 北京、上海、深圳三大城市门店数量占总数的{city_counts.head(3).sum()/city_counts.sum()*100:.1f}%
4. **月度趋势明显**: 主要渠道在3月份销售额普遍较高，可能与季节性消费特征相关

## 5. 建议与启示

1. **渠道优化**: 重点扶持高效率渠道，优化低效渠道的资源配置
2. **区域拓展**: 在门店密度较低但消费潜力大的城市加大布局
3. **精准营销**: 根据不同渠道的销售特点制定差异化营销策略
4. **库存管理**: 参考月度销售趋势，合理安排库存水平
"""

# 保存报告（Mac路径适配）
report_path = '金佰利数据统计分析报告_中文完整版.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"✅ 统计分析报告已生成完成！")
print(f"📁 报告保存路径: {report_path}")
print(f"\n📋 生成文件清单:")
print(f"1. 深度分析图表（中文舒展版）: {save_path}")
print(f"2. 统计分析报告（中文完整版）: {report_path}")

各大区门店与销售统计
    门店数量  年度销售总额(万元)  单店平均销售额(万元)
大区                               
东区   874    25974.52        29.72
北区   694    18907.88        27.24
南区   583    17747.41        30.44
西区   342     8210.98        24.01

主要城市-渠道销售交叉分析（单位：万元）
渠道    GT-MT      华润      大润发       永辉     沃尔玛
地城市                                          
上海     0.00    0.00  1165.39   433.02  249.82
佛山     0.00   78.42   321.42    94.09  300.79
北京   531.31    0.00   170.71  1621.45  520.54
南京    13.09    0.00   367.99   189.42  133.08
广州     0.00  452.66   182.93   285.99  404.89
成都     0.00    0.00   176.70   467.76  234.30
杭州     0.00  157.33   137.94   731.25  166.46
武汉     0.00    0.00   182.66     0.00  451.85
深圳     0.00  484.35    84.76   248.37  514.65
重庆     0.00    0.00    33.13  1311.16  119.22

✅ 深度分析图表（中文舒展版）已生成完成！
📁 深度分析图表保存路径: 金佰利大区城市渠道深度分析_中文舒展版.png


NameError: name 'city_percent' is not defined